## 1. Imports

In [1]:
import os
import json
import time
import random
import torch
import tiktoken
import shutil
from typing import List

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()
print("Imports OK")
print(f"GPU available: {torch.cuda.is_available()}")

C:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK
GPU available: True


## 2. Paths

In [2]:
PDF_DIR = "./clinical_pdfs/"
DB_DIR = "./db_wound_care"

if not os.path.exists(PDF_DIR):
    os.makedirs(PDF_DIR)
    print(f"Created {PDF_DIR} — place your clinical PDFs here.")
else:
    pdfs = [f for f in os.listdir(PDF_DIR) if f.endswith(".pdf")]
    print(f"PDF_DIR ready — {len(pdfs)} PDF(s) found: {pdfs}")

PDF_DIR ready — 2 PDF(s) found: ['Book2-wound-dressing-guide.pdf', 'HDFT-Wound-Dressing-Guideline-2018-v2.1-PDF.pdf']


## 3. Rate limit helpers

In [3]:
def count_tokens(text: str, model: str = "gpt-4o-mini") -> int:
    """Estimate tokens in a string using tiktoken before sending to OpenAI."""
    try:
        enc = tiktoken.encoding_for_model(model)
    except KeyError:
        enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(text))
 
 
def call_with_backoff(fn, max_retries: int = 6, base_delay: float = 1.0):
    """
    Calls fn() and retries on 429 rate-limit errors using exponential backoff.
    Parses OpenAI's suggested retry delay when available.
    """
    for attempt in range(max_retries):
        try:
            return fn()
        except Exception as e:
            err_str = str(e)
            is_rate_limit = "429" in err_str or "rate_limit_exceeded" in err_str
 
            if is_rate_limit and attempt < max_retries - 1:
                retry_ms = None
                if "Please try again in" in err_str:
                    try:
                        after_part = err_str.split("Please try again in")[1]
                        ms_str = after_part.strip().split("ms")[0].strip()
                        retry_ms = float(ms_str) / 1000.0
                    except Exception:
                        pass
 
                if retry_ms:
                    wait = retry_ms + 0.5
                    print(f"     ⏳ Rate limit — OpenAI says wait {retry_ms:.1f}s, sleeping {wait:.1f}s...")
                else:
                    wait = base_delay * (2 ** attempt) * (0.8 + random.random() * 0.4)
                    print(f"     ⏳ Rate limit (attempt {attempt+1}/{max_retries}) — sleeping {wait:.1f}s...")
 
                time.sleep(wait)
            else:
                raise  # non-rate-limit error or out of retries
 
    raise RuntimeError(f"Max retries ({max_retries}) exceeded")
 
 
class TPMThrottle:
    """
    Proactively tracks tokens-per-minute and sleeps before hitting the limit.
    Much better than reacting to 429s after they happen.
    """
    def __init__(self, tpm_limit: int = 200_000, safety_margin: float = 0.85):
        self.tpm_limit     = tpm_limit
        self.safety_margin = safety_margin
        self.window_start  = time.time()
        self.tokens_used   = 0
 
    def _reset_if_new_window(self):
        if time.time() - self.window_start >= 60.0:
            self.window_start = time.time()
            self.tokens_used  = 0
 
    def check_and_wait(self, tokens_needed: int):
        """Sleep until next window if this request would exceed the safe ceiling."""
        self._reset_if_new_window()
        soft_limit = self.tpm_limit * self.safety_margin
        if self.tokens_used + tokens_needed > soft_limit:
            elapsed   = time.time() - self.window_start
            remaining = 60.0 - elapsed + 1.0   # +1s buffer
            if remaining > 0:
                print(f"     🕐 TPM ceiling reached ({self.tokens_used:,} / {int(soft_limit):,} tokens). "
                      f"Waiting {remaining:.1f}s for new window...")
                time.sleep(remaining)
                self.window_start = time.time()
                self.tokens_used  = 0
 
    def record(self, tokens_used: int):
        """Record tokens consumed after a successful API call."""
        self._reset_if_new_window()
        self.tokens_used += tokens_used
 
 
# One shared throttle instance for the entire pipeline run
throttle = TPMThrottle(tpm_limit=200_000, safety_margin=0.85)
print("Rate limit helpers ready")
 

Rate limit helpers ready


## 4. PDF partitioning & chunking

In [4]:
def partition_document(file_path: str):
    """Extract text, tables and images from a PDF using unstructured hi_res."""
    print(f"📄 Partitioning: {file_path}")
    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True,
    )
    print(f"   ✅ Extracted {len(elements)} elements")
    return elements

def create_chunks_by_title(elements):
    """Chunk elements by document title/section boundaries."""
    print("🔨 Creating smart chunks...")
    chunks = chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500,
    )
    print(f"   ✅ Created {len(chunks)} chunks")
    return chunks

## 5. Content separation & AI summarisation

In [5]:
def separate_content_types(chunk):
    """Split a chunk into its text, table and image components."""
    content_data = {
        "text":   chunk.text,
        "tables": [],
        "images": [],
        "types":  ["text"],
    }
    if hasattr(chunk, "metadata") and hasattr(chunk.metadata, "orig_elements"):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            if element_type == "Table":
                content_data["types"].append("table")
                table_html = getattr(element.metadata, "text_as_html", element.text)
                content_data["tables"].append(table_html)
            elif element_type == "Image":
                if hasattr(element, "metadata") and hasattr(element.metadata, "image_base64"):
                    content_data["types"].append("image")
                    content_data["images"].append(element.metadata.image_base64)
    content_data["types"] = list(set(content_data["types"]))
    return content_data
 
 
def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """
    Send text + tables + images to GPT-4o-mini for a clinical summary.
    Uses TPMThrottle (proactive) + call_with_backoff (reactive) for rate limits.
    Falls back gracefully to raw text if all retries are exhausted.
    """
    try:
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
 
        prompt_text = (
            "You are a clinical wound care expert. Analyze this guideline content:\n\n"
            f"TEXT CONTENT:\n{text}\n\n"
        )
        if tables:
            prompt_text += "TABLES (HTML):\n" + "\n".join(tables) + "\n"
        prompt_text += (
            "\nTASK: Summarize the clinical guideline content into a concise, "
            "searchable chunk focused on wound care and dressing recommendations."
        )
 
        message_content = [{"type": "text", "text": prompt_text}]
        for image_b64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
            })
 
        # Proactive throttle check before sending
        tokens_needed = count_tokens(prompt_text) + 300   # +300 for response
        throttle.check_and_wait(tokens_needed)
 
        # Reactive backoff if 429 still occurs
        def api_call():
            return llm.invoke([HumanMessage(content=message_content)])
 
        response = call_with_backoff(api_call)
        throttle.record(tokens_needed)
        return response.content
 
    except Exception as e:
        print(f"     ❌ AI summary failed after retries: {e}")
        # Graceful fallback — raw text so the chunk is still indexed
        fallback = text[:500]
        if tables:
            fallback += f"\n[Contains {len(tables)} table(s) with dressing data]"
        if images:
            fallback += f"\n[Contains {len(images)} clinical image(s)]"
        return fallback
 
 
def summarise_chunks(chunks):
    """Process all chunks — AI summary for mixed content, raw text for text-only."""
    print("🧠 Processing chunks with AI Summaries...")
    langchain_documents = []
    total = len(chunks)
 
    for i, chunk in enumerate(chunks, start=1):
        print(f"   Processing chunk {i}/{total}")
        content_data = separate_content_types(chunk)
        print(f"     Types: {content_data['types']} | "
              f"Tables: {len(content_data['tables'])} | "
              f"Images: {len(content_data['images'])}")
 
        if content_data["tables"] or content_data["images"]:
            print("     → Creating AI summary for mixed content...")
            enhanced = create_ai_enhanced_summary(
                content_data["text"],
                content_data["tables"],
                content_data["images"],
            )
            print(f"     → Preview: {enhanced[:200]}...")
        else:
            print("     → Using raw text (no tables/images)")
            enhanced = content_data["text"]
 
        doc = Document(
            page_content=enhanced,
            metadata={
                "original_content": json.dumps({
                    "raw_text":      content_data["text"],
                    "tables_html":   content_data["tables"],
                    "images_base64": content_data["images"],
                })
            },
        )
        langchain_documents.append(doc)
 
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents

## 6. T.I.M.E. tag extractor

In [6]:
def extract_time_tags(text: str) -> str:
    """
    Scan chunk text for T.I.M.E. keywords.
    Returns comma-separated string (e.g. "T,M") or "none".
    ChromaDB does NOT accept empty lists in metadata — string is safer.
    """
    tags = []
    t_lower = text.lower()
    if any(w in t_lower for w in ["tissue", "necrotic", "slough", "granulation", "eschar"]):
        tags.append("T")
    if any(w in t_lower for w in ["infection", "biofilm", "bacteria", "antimicrobial", "sepsis"]):
        tags.append("I")
    if any(w in t_lower for w in ["moisture", "exudate", "maceration", "dry", "drainage", "absorbent"]):
        tags.append("M")
    if any(w in t_lower for w in ["edge", "epithelial", "advancing", "undermining", "periwound"]):
        tags.append("E")
    return ",".join(tags) if tags else "none"
 

## 7. Vector store creation

In [7]:
def create_vector_store(documents, persist_directory):
    """Embed all documents with MedEmbed and persist to ChromaDB."""
    print("🔮 Creating embeddings and storing in ChromaDB...")
 
    embedding_model = HuggingFaceEmbeddings(
        model_name="abhinand/MedEmbed-large-v0.1",
        model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
 
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"},
    )
    print("--- Finished ---")
    print(f"✅ Vector store saved to {persist_directory}")
    return vectorstore
 
 
def get_pdf_files(directory: str) -> List[str]:
    pdf_files = [
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if f.lower().endswith(".pdf")
    ]
    print(f"📂 Found {len(pdf_files)} PDF(s) in {directory}")
    return pdf_files

## 8. Main pipeline

In [8]:
def run_folder_ingestion_pipeline(folder_path: str, persist_directory: str):
    """
    Full ingestion pipeline:
      1. Find all PDFs
      2. Partition each PDF (text + tables + images)
      3. Chunk by title/section
      4. AI-summarise mixed-content chunks (with rate limit handling)
      5. Tag each chunk with T.I.M.E. keywords
      6. Embed with MedEmbed-large and store in ChromaDB
    """
    print("🚀 Starting MULTI-DOCUMENT RAG Ingestion Pipeline")
    print("=" * 50)
 
    pdf_paths = get_pdf_files(folder_path)
    if not pdf_paths:
        print("❌ No PDFs found. Exiting.")
        return None
 
    all_langchain_docs = []
 
    for path in pdf_paths:
        print(f"\n--- Processing: {os.path.basename(path)} ---")
 
        elements  = partition_document(path)
        chunks    = create_chunks_by_title(elements)
        file_docs = summarise_chunks(chunks)
 
        for chunk_index, doc in enumerate(file_docs):
            doc.metadata["source"]      = os.path.basename(path)
            doc.metadata["chunk_index"] = chunk_index
            doc.metadata["time_tags"]   = extract_time_tags(doc.page_content)
 
        all_langchain_docs.extend(file_docs)
        print(f"   Added {len(file_docs)} chunks from {os.path.basename(path)}")
 
    print(f"\n🔮 Indexing {len(all_langchain_docs)} total chunks into ChromaDB...")
    db = create_vector_store(all_langchain_docs, persist_directory=persist_directory)
 
    print("\n🎉 Folder ingestion completed successfully!")
    print(f"   Total chunks indexed: {len(all_langchain_docs)}")
    return db

In [9]:
db = run_folder_ingestion_pipeline(PDF_DIR, DB_DIR)

🚀 Starting MULTI-DOCUMENT RAG Ingestion Pipeline
📂 Found 2 PDF(s) in ./clinical_pdfs/

--- Processing: Book2-wound-dressing-guide.pdf ---
📄 Partitioning: ./clinical_pdfs/Book2-wound-dressing-guide.pdf


Loading weights: 100%|█████████████████████████████████████████████████████████████| 367/367 [00:00<00:00, 5251.95it/s]


   ✅ Extracted 898 elements
🔨 Creating smart chunks...
   ✅ Created 63 chunks
🧠 Processing chunks with AI Summaries...
   Processing chunk 1/63
     Types: ['text', 'image', 'table'] | Tables: 3 | Images: 4
     → Creating AI summary for mixed content...
     → Preview: ### Wound Dressing Guide Summary

**Purpose:** This guide aims to promote healthy skin and effective wound care practices, particularly in aged care settings.

**Funding:** Supported by the Australian...
   Processing chunk 2/63
     Types: ['text', 'image'] | Tables: 0 | Images: 1
     → Creating AI summary for mixed content...
     → Preview: ### Wound Care and Dressing Recommendations

**Purpose:** This guideline provides an overview of wound dressing products aimed at optimizing the local wound environment to promote healing.

**Key Reas...
   Processing chunk 3/63
     Types: ['text', 'image'] | Tables: 0 | Images: 1
     → Creating AI summary for mixed content...
     → Preview: ### Wound Dressing Guidelines Summa

Loading weights: 100%|█████████████████████████████████████████████████████████████| 391/391 [00:00<00:00, 8012.61it/s]


--- Creating vector store ---
--- Finished ---
✅ Vector store saved to ./db_wound_care

🎉 Folder ingestion completed successfully!
   Total chunks indexed: 106
